# Étape 17c — Simulation : que se passe-t-il quand on connaît les vrais prérequis ?

**Objectif.** Sur Junyi, on ne peut pas savoir si les prérequis des experts influencent vraiment la réussite des élèves. Ici, on **fabrique** des élèves dont la réussite dépend, de façon contrôlée, d'un graphe de prérequis **connu**.
On peut alors distinguer ce qui vient **des données** (y a-t-il un effet de prérequis exploitable ?) et ce qui vient **des modèles** (savent-ils l'utiliser ?).

**Le générateur est différent de nos modèles** (exigence de l'encadrant) : c'est un modèle de type IRT avec apprentissage, et non un réseau de neurones.
Chaque élève a une aptitude générale et une aptitude par thème. Pour chaque notion, sa réussite dépend de sa maîtrise de la notion **et** de sa maîtrise des prérequis (force λ) ; l'apprentissage d'une notion est freiné si ses prérequis ne sont pas maîtrisés.

**Domaine simulé :** 120 notions (1 exercice par notion), 6 thèmes de 20, un graphe de prérequis orienté sans cycle (1 à 3 prérequis par notion, 85 % dans le même thème). Le même graphe sert aux trois scénarios ; 3 000 élèves par scénario.

| Scénario | Effet des prérequis | Ordre de travail | Question |
|---|---|---|---|
| **S1** prérequis forts | fort (λ = 2) | libre | Quand les prérequis comptent vraiment, la justesse et le sens des flèches comptent-ils davantage ? Les flèches apprises retrouvent-elles les prérequis ? |
| **S2** prérequis faibles | faible (λ = 0,7) | libre | Que devient la récupération quand l'effet est faible ? |
| **S3** ordre imposé | **aucun** (λ = 0) | suit le programme | Les méthodes confondent-elles **ordre du programme** et **prérequis** ? |

**Pour chaque scénario** (5 % et 20 % des élèves d'entraînement, **5 graines**) :
1. **Utilité pour le KT** (PrereqKT, AUC des 1res tentatives) : `aucun`, `expert` 25 % et 100 %, `theme_faux` 100 %, `hors_theme_faux` 100 %, et **`inverse` 100 %** (pour chaque notion, ses *successeurs* au lieu de ses prérequis : bonnes notions, **mauvais sens**) ;
2. **Récupération des prérequis** (AUC face au vrai graphe, sur les paires d'un même thème) : flèches **apprises** (modèle de l'étape 15, sans prior), **ordre de passage** (qui fait A avant B),
   **co-réussite** (« lift » : réussir A augmente-t-il la réussite de B ? — méthode proche de celle de Truong, 2026), et **pratique avant** (les élèves qui ont travaillé A *avant* B réussissent-ils mieux B ?).

Deux comptes en parallèle : `PARTIE = 'A'` (graines 0, 1, 2) et `PARTIE = 'B'` (graines 3, 4). **GPU T4.**

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule S0 — Génération du domaine et des élèves simulés (une seule fois, quelques minutes, sans GPU)
SIM = os.path.join(DOSSIER, 'simulation')
SCENARIOS = {'S1': dict(lam=2.0, ordre='libre'), 'S2': dict(lam=0.7, ordre='libre'), 'S3': dict(lam=0.0, ordre='impose')}
N_NOTIONS, N_THEMES, N_ELEVES, MAX_REP = 120, 6, 3000, 150

def sigm(x):
    return 1 / (1 + np.exp(-x))

# 1) Domaine : thèmes, difficultés, graphe de prérequis orienté sans cycle (prérequis = notions d'indice plus petit)
rng_d = np.random.default_rng(2026)
theme = np.arange(N_NOTIONS) // (N_NOTIONS // N_THEMES)
difficulte = rng_d.normal(0, 0.8, N_NOTIONS)
prereq = {k: [] for k in range(N_NOTIONS)}
for k in range(N_NOTIONS):
    meme = [a for a in range(k) if theme[a] == theme[k]]
    autres = [a for a in range(k) if theme[a] != theme[k]]
    if not meme and not autres:
        continue
    for _ in range(int(rng_d.integers(1, 4))):
        pool = meme if (meme and (rng_d.random() < 0.85 or not autres)) else autres
        a = int(rng_d.choice(pool))
        if a not in prereq[k]:
            prereq[k].append(a)
A_vrai = np.zeros((N_NOTIONS, N_NOTIONS), np.float32)
for k, ps in prereq.items():
    for a in ps:
        A_vrai[a, k] = 1

# 2) Élèves
def simuler(lam, ordre, graine):
    rng = np.random.default_rng(graine)
    seqs = []
    for u in range(N_ELEVES):
        apt = rng.normal(0, 1); apt_t = rng.normal(0, 0.7, N_THEMES)
        m = np.zeros(N_NOTIONS); vues = np.zeros(N_NOTIONS, bool)
        q_, r_ = [], []
        while len(q_) < MAX_REP and not vues.all():
            restantes = np.flatnonzero(~vues)
            if ordre == 'impose':      # suit le programme (ordre des indices), avec un peu de bruit
                k = restantes[0] if rng.random() < 0.85 else rng.choice(restantes[:6])
            else:                      # ordre libre : une notion non encore vue, au hasard
                k = rng.choice(restantes)
            vues[k] = True
            for essai in range(3):     # jusqu'à 3 essais, on passe à la suite dès la réussite
                pm = m[prereq[k]].mean() if prereq[k] else 1.0
                p = sigm(0.8 + apt + apt_t[theme[k]] - difficulte[k] + 2.0 * m[k] - lam * (1 - pm))
                ok = rng.random() < p
                q_.append(int(k)); r_.append(int(ok))
                porte = (0.3 + 0.7 * pm) if lam > 0 else 1.0          # apprendre est plus lent si les prérequis ne sont pas maîtrisés
                m[k] = min(1.0, m[k] + 0.5 * porte * (1.2 if ok else 1.0))
                if ok or len(q_) >= MAX_REP:
                    break
        seqs.append((f'e{u}', q_, r_))
    return seqs

rng_split = np.random.default_rng(42)
parties_ = rng_split.permutation(N_ELEVES)
partie_de = {f'e{u}': ('train' if i < 0.7 * N_ELEVES else 'valid' if i < 0.8 * N_ELEVES else 'test') for i, u in enumerate(parties_)}
for sc, par in SCENARIOS.items():
    d_ = os.path.join(SIM, sc); os.makedirs(d_, exist_ok=True)
    if os.path.exists(os.path.join(d_, 'sequences_test.csv')):
        print(sc, ': déjà généré'); continue
    seqs = simuler(par['lam'], par['ordre'], graine=100 + int(sc[1]))
    lignes = pd.DataFrame([{'uid': u, 'partie': partie_de[u], 'morceau': 0, 'exercices': ','.join(map(str, q_)),
                            'reponses': ','.join(map(str, r_))} for u, q_, r_ in seqs if len(q_) >= 2])
    for p_ in ['train', 'valid', 'test']:
        lignes[lignes.partie == p_].drop(columns='partie').to_csv(os.path.join(d_, f'sequences_{p_}.csv'), index=False)
    pd.DataFrame({'exo_id': range(N_NOTIONS), 'name': [f'notion_{k}' for k in range(N_NOTIONS)],
                  'topic': [f'theme_{t}' for t in theme]}).to_csv(os.path.join(d_, 'exercices.csv'), index=False)
    np.save(os.path.join(d_, 'adjacence_expert.npy'), A_vrai)
    toutes_r = np.concatenate([np.array(x.split(','), int) for x in lignes.reponses])
    print(f'{sc} : {len(lignes)} élèves | {len(toutes_r):,} réponses | taux de réussite {toutes_r.mean():.2f}')
print(f'Graphe vrai : {int(A_vrai.sum())} prérequis | {100 * np.mean([theme[a] == theme[b] for a, b in zip(*np.nonzero(A_vrai))]):.0f} % dans le même thème')
PREP = os.path.join(SIM, 'S1')     # les cellules suivantes chargent d'abord S1 (même graphe et mêmes exercices pour S2 et S3)

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule S1 — Construction des graphes factoriels (simulation) (déterministe pour chaque graine)
COUVERTURES = [0.25, 0.5, 1.0]
SOURCES = ['expert', 'theme_faux', 'hors_theme_faux']
GRAINES = list(range(10))

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
ancetres = {b: nx.ancestors(G_exp, b) for b in range(N)}
prereq_directs = {b: list(G_exp.predecessors(b)) for b in range(N)}
UNIVERS = sorted(b for b in range(N) if prereq_directs[b])
topic = exos.set_index('exo_id')['topic'].to_dict()
meme_theme = {}
for b in range(N):
    meme_theme[b] = [a for a in range(N) if a != b and isinstance(topic[b], str) and topic[a] == topic[b]]

def graphe_factoriel(couv, source, graine):
    rng = random.Random(90_000 + graine)
    ordre = UNIVERS[:]; rng.shuffle(ordre)                 # sous-ensembles emboîtés : 25 % ⊂ 50 % ⊂ 100 %
    choisis = ordre[:int(round(couv * len(ordre)))]
    E, manquants = [], 0
    for b in sorted(choisis):
        n_b = len(prereq_directs[b])
        if source == 'expert':
            E += [(a, b) for a in prereq_directs[b]]
            continue
        interdits = ancetres[b] | {b}
        if source == 'theme_faux':
            pool = [a for a in meme_theme[b] if a not in interdits]
        else:
            pool = [a for a in range(N) if a not in interdits and (not isinstance(topic[b], str) or topic[a] != topic[b])]
        rng_b = random.Random(f'{graine}-{source}-{b}')        # même tirage pour B à toutes les couvertures
        pris = rng_b.sample(pool, min(n_b, len(pool)))
        manquants += n_b - len(pris)
        E += [(a, b) for a in pris]
    return sorted(set(E)), manquants

# Vérifications : nombre de flèches, part des flèches entre exercices du même thème, couverture des premières tentatives
_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values
def couverture_1res(E):
    a = np.zeros(N, bool)
    for _, b in E:
        a[b] = True
    return 100 * (poids * a).sum() / poids.sum()

lignes = []
for couv in COUVERTURES:
    for source in SOURCES:
        m = []
        for g in GRAINES:
            E, manq = graphe_factoriel(couv, source, g)
            meme = np.mean([isinstance(topic[b], str) and topic[a] == topic[b] for a, b in E]) if E else 0
            juste = np.mean([a in ancetres[b] for a, b in E]) if E else 0
            m.append({'fleches': len(E), 'manquantes': manq, 'meme_theme_%': 100 * meme,
                      'prerequis_vrais_%': 100 * juste, 'couverture_1res_%': couverture_1res(E)})
        lignes.append({'couverture': couv, 'source': source, **pd.DataFrame(m).mean().round(1).to_dict()})
verif = pd.DataFrame(lignes)
verif.to_csv(os.path.join(RESULTATS, 'sim_factoriel_graphes.csv'), index=False)
print(f'Univers : {len(UNIVERS)} exercices avec au moins un prérequis expert')
print(verif.to_string(index=False))

In [ ]:
# Cellule S2 — Le modèle à flèches apprises (repris de l'étape 15)
HP_APPRIS = dict(HP, lam_l1=0.01, lam_acyc=0.01, lr_aretes=1e-2, poids_depart=0.1)   # lam_acyc faible : avec le prior LLM, la pénalité de départ est grande

class PrereqKTAppris(GraphKT):
    def __init__(self, n, hp, paires, logits_depart):
        super().__init__(n, hp, np.zeros((n, n), np.float32), indicateurs=True)
        self.register_buffer('src', torch.as_tensor(paires[:, 0]))
        self.register_buffer('dst', torch.as_tensor(paires[:, 1]))
        self.theta = nn.Parameter(torch.as_tensor(logits_depart, dtype=torch.float32))

    def poids(self):
        return torch.sigmoid(self.theta)

    def W(self):                                   # W[a, b] = poids de la flèche a -> b
        W = torch.zeros(self.n, self.n, device=self.theta.device)
        return W.index_put((self.src, self.dst), self.poids())

    def forward(self, q, r):
        W = self.W()
        P_pre = W.T / (W.T.sum(1, keepdim=True) + 1e-6)     # ligne j : prérequis de j (pondérés)
        P_suc = W / (W.sum(1, keepdim=True) + 1e-6)         # ligne i : successeurs de i
        a_pre = 1 - torch.exp(-W.T.sum(1, keepdim=True)); a_suc = 1 - torch.exp(-W.sum(1, keepdim=True))
        H = self.E.weight
        for c in self.couches:
            H = H + torch.relu(c['pre'](P_pre @ H)) * a_pre + torch.relu(c['suc'](P_suc @ H)) * a_suc
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        return self.pred(torch.cat([self.drop(h), H[q[:, 1:]], self.indicateurs_doux(q, r, W)], dim=-1)).squeeze(-1)

    def indicateurs_doux(self, q, r, W):
        # mêmes 6 indicateurs que PrereqKT, mais chaque voisin compte pour le poids de sa flèche
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float(); taux = reussites / essais.clamp(min=1)
        sortie = []
        for M in (W.T, W):
            v = M[q[:, 1:]]
            n_v = v.sum(-1); n_t = (v * tente).sum(-1)
            sortie += [(v * tente * taux).sum(-1) / (n_t + 1e-6), n_t / (n_v + 1e-6), 1 - torch.exp(-n_v)]
        return torch.stack(sortie, -1)

    def acyclicite(self):                          # NOTEARS : tr(exp(W∘W)) − n, nul si et seulement si le graphe est sans cycle
        W = self.W()
        return torch.trace(torch.matrix_exp(W * W)) - self.n

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 0.01, 0.99)
    return np.log(p / (1 - p))

def entrainer_appris(depart, fraction, graine, hp=HP_APPRIS):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    if depart == 'sans_prior':
        l0 = logit(hp['poids_depart']) + np.random.default_rng(graine).normal(0, 0.01, len(PAIRES))
    else:
        l0 = logit(cand.p_oui.values)
    modele = PrereqKTAppris(N, hp, PAIRES, l0).to(DEVICE)
    autres = [p for n_, p in modele.named_parameters() if n_ != 'theta']
    opt = torch.optim.Adam([{'params': autres, 'lr': hp['lr']}, {'params': [modele.theta], 'lr': hp['lr_aretes']}])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float()) \
                + hp['lam_l1'] * modele.poids().mean() + hp['lam_acyc'] * modele.acyclicite()
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    with torch.no_grad():
        poids = modele.poids().cpu().numpy()
        acyc = float(modele.acyclicite())
    return ({'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
             'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1), 'acyclicite': round(acyc, 4),
             'poids_moyen': round(float(poids.mean()), 4), 'fleches_poids_sup_0.5': int((poids > 0.5).sum())}, poids)

In [ ]:
# Cellule S3 — Outils : condition « inverse », chargement d'un scénario, mesures de récupération des prérequis
def graphe_inverse(couv, graine):
    # pour chaque notion B choisie : autant de sources que de vrais prérequis, prises parmi ses SUCCESSEURS (puis descendants)
    E, _ = graphe_factoriel(couv, 'expert', graine)
    rng = random.Random(f'inv-{graine}')
    sortie = []
    for b in sorted({b for _, b in E}):
        n_b = len(prereq_directs[b])
        desc = list(G_exp.successors(b)) + [x for x in nx.descendants(G_exp, b) if x not in set(G_exp.successors(b))]
        pris = desc[:n_b]
        if len(pris) < n_b:
            pool = [a for a in meme_theme[b] if a not in ancetres[b] and a not in pris]
            pris += rng.sample(pool, min(n_b - len(pris), len(pool)))
        sortie += [(a, b) for a in pris]
    return sorted(set(sortie))

# Paires candidates : toutes les paires ordonnées d'un même thème (le modèle appris choisit parmi elles)
cand = pd.DataFrame([(a, b) for a in range(N) for b in range(N) if a != b and topic[a] == topic[b]], columns=['A', 'B'])
cand['p_oui'] = 0.5
cand['expert'] = A_expert[cand.A, cand.B] > 0
cand['ancetre'] = [a in ancetres[b] for a, b in zip(cand.A, cand.B)]
PAIRES = cand[['A', 'B']].values.astype(np.int64)
SOUS_ENS = {'toutes': np.ones(len(cand), bool)}

def qualite(score, prefixe):
    score = np.asarray(score, dtype=float); y = cand.expert.values; K = int(y.sum())
    return {f'{prefixe}AUC': round(roc_auc_score(y, score), 4), f'{prefixe}AUC_etendue': round(roc_auc_score(cand.ancetre.values, score), 4),
            f'{prefixe}precisionK': round(float(y[np.argsort(-score, kind='stable')[:K]].mean()), 4)}

def charger_scenario(sc):
    global PREP, donnees, T_valid, T_test
    PREP = os.path.join(SIM, sc)
    donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
    T_valid = tenseurs(*donnees['valid'][1:]); T_test = tenseurs(*donnees['test'][1:])

def stats_paires(fraction, graine):
    # ordre de passage (A avant B) et co-réussite (réussir A en 1re tentative, avant B, change-t-il la réussite de B ?)
    u, q, r, f = donnees['train']
    if fraction < 1:
        rng = np.random.default_rng(graine); el = np.unique(u)
        gardes = set(rng.choice(el, size=max(1, int(fraction * len(el))), replace=False))
    else:
        gardes = set(u)
    A_, B_ = PAIRES[:, 0], PAIRES[:, 1]
    n_deux = np.zeros(len(PAIRES)); n_avant = np.zeros(len(PAIRES))
    s1 = np.zeros(len(PAIRES)); n1 = np.zeros(len(PAIRES)); s0 = np.zeros(len(PAIRES)); n0 = np.zeros(len(PAIRES))
    sa = np.zeros(len(PAIRES)); sp = np.zeros(len(PAIRES))      # réussite de B (1re tentative) quand A a été pratiqué avant / après B
    for uid, qq, rr, ff in zip(u, q, r, f):
        if uid not in gardes:
            continue
        pos = np.full(N, np.inf); r1 = np.zeros(N)
        idx = np.flatnonzero(ff); pos[qq[idx]] = idx; r1[qq[idx]] = rr[idx]
        vu = np.isfinite(pos[A_]) & np.isfinite(pos[B_])
        av = vu & (pos[A_] < pos[B_])
        n_deux += vu; n_avant += av
        s1 += av * (r1[A_] == 1) * r1[B_]; n1 += av * (r1[A_] == 1)
        s0 += av * (r1[A_] == 0) * r1[B_]; n0 += av * (r1[A_] == 0)
        sa += av * r1[B_]; sp += (vu & ~av) * r1[B_]
    ordre = (n_avant + 1) / (n_deux + 2)
    lift = (s1 + 1) / (n1 + 2) - (s0 + 1) / (n0 + 2)
    pratique = (sa + 1) / (n_avant + 2) - (sp + 1) / (n_deux - n_avant + 2)
    return ordre, lift, pratique

charger_scenario('S1')
print(f'{len(cand)} paires candidates (même thème), dont {int(cand.expert.sum())} vrais prérequis')

In [ ]:
# Cellule S4 — Les entraînements de votre partie (reprise automatique)
PARTIE = 'A'          # 'A' = graines 0, 1, 2 ; 'B' = graines 3, 4 (sur l'autre compte, en même temps)

GRAINES_PARTIE = {'A': [0, 1, 2], 'B': [3, 4]}[PARTIE]
FRACTIONS = [0.2, 0.05]
CONDS = [('aucun', None, None), ('expert_c25', 0.25, 'expert'), ('expert_c100', 1.0, 'expert'),
         ('theme_faux_c100', 1.0, 'theme_faux'), ('hors_theme_faux_c100', 1.0, 'hors_theme_faux'), ('inverse_c100', 1.0, 'inverse')]
chemin_s = os.path.join(RESULTATS, f'sim_resultats_{PARTIE}.csv')
res_s = pd.read_csv(chemin_s).to_dict('records') if os.path.exists(chemin_s) else []
fait = set()
for p_ in ['A', 'B']:
    c_ = os.path.join(RESULTATS, f'sim_resultats_{p_}.csv')
    if os.path.exists(c_):
        fait |= {(x['scenario'], x['condition'], float(x['fraction']), int(x['graine'])) for x in pd.read_csv(c_).to_dict('records')}

def noter(ligne):
    res_s.append(ligne); pd.DataFrame(res_s).to_csv(chemin_s, index=False)

t0_ = time.time()
for sc in SCENARIOS:
    charger_scenario(sc)
    for g_ in GRAINES_PARTIE:
        for frac in FRACTIONS:
            if (sc, 'ordre_lift', frac, g_) not in fait:
                o_, l_, pr_ = stats_paires(frac, g_)
                noter({'scenario': sc, 'condition': 'ordre_lift', 'fraction': frac, 'graine': g_,
                       **qualite(o_, 'ordre_'), **qualite(l_, 'lift_'), **qualite(pr_, 'pratique_')})
            for lib, couv, source in CONDS:
                if (sc, lib, frac, g_) in fait:
                    continue
                if couv is None:
                    A = graphe('aucun')
                elif source == 'inverse':
                    A = vers_matrice(graphe_inverse(couv, g_))
                else:
                    A = vers_matrice(graphe_factoriel(couv, source, g_)[0])
                res = entrainer('PrereqKT', A, fraction=frac, graine=g_)
                noter({'scenario': sc, 'condition': lib, 'fraction': frac, 'graine': g_, **res})
                print(f'  {sc} | graine {g_} | {frac:.0%} | {lib:22s} | AUC 1res = {res["AUC_1re_test"]} | {res["minutes"]} min')
            if (sc, 'appris_sans_prior', frac, g_) not in fait:
                res, poids = entrainer_appris('sans_prior', frac, g_)
                q_ = qualite(poids, 'appris_')
                noter({'scenario': sc, 'condition': 'appris_sans_prior', 'fraction': frac, 'graine': g_, **res, **q_})
                print(f'  {sc} | graine {g_} | {frac:.0%} | appris_sans_prior      | AUC 1res = {res["AUC_1re_test"]} | AUC flèches = {q_["appris_AUC"]}')
        # récupération des prérequis avec 100 % des élèves (pas d'expérience KT à 100 %)
        if (sc, 'ordre_lift', 1.0, g_) not in fait:
            o_, l_, pr_ = stats_paires(1.0, g_)
            noter({'scenario': sc, 'condition': 'ordre_lift', 'fraction': 1.0, 'graine': g_,
                   **qualite(o_, 'ordre_'), **qualite(l_, 'lift_'), **qualite(pr_, 'pratique_')})
        if (sc, 'appris_sans_prior', 1.0, g_) not in fait:
            res, poids = entrainer_appris('sans_prior', 1.0, g_)
            noter({'scenario': sc, 'condition': 'appris_sans_prior', 'fraction': 1.0, 'graine': g_, **res, **qualite(poids, 'appris_')})
    print(f'Scénario {sc} terminé ({(time.time() - t0_) / 60:.0f} min écoulées)')
print(f'Partie {PARTIE} terminée.')

In [ ]:
# Cellule S5 — Analyse (quand les deux parties sont finies) ; après une déconnexion : cellules 1 à S3, puis celle-ci
import matplotlib.pyplot as plt
col = 'AUC_1re_test'
tab = pd.concat([pd.read_csv(os.path.join(RESULTATS, f'sim_resultats_{p}.csv')) for p in ['A', 'B']
                 if os.path.exists(os.path.join(RESULTATS, f'sim_resultats_{p}.csv'))], ignore_index=True)
tab = tab.drop_duplicates(['scenario', 'condition', 'fraction', 'graine'])
n_ = tab.groupby('graine').size()
graines_ok = sorted(n_[n_ == len(SCENARIOS) * (2 * 8 + 2)].index)
tab = tab[tab.graine.isin(graines_ok)]
print(f'Graines complètes analysées : {graines_ok}')

def ic_bootstrap(d, n=10_000, graine=0):
    rng = np.random.default_rng(graine); d = np.asarray(d, dtype=float)
    return np.percentile(rng.choice(d, size=(n, len(d)), replace=True).mean(1), [2.5, 97.5])

LIBS = ['expert_c25', 'expert_c100', 'theme_faux_c100', 'hors_theme_faux_c100', 'inverse_c100', 'appris_sans_prior']
synthese, hyp, recup = [], [], []
for sc in SCENARIOS:
    for frac in FRACTIONS:
        s = tab[(tab.scenario == sc) & (tab.fraction == frac)]
        piv = s.pivot_table(index='graine', columns='condition', values=col)
        base = piv['aucun']
        for lib in LIBS:
            d = piv[lib] - base; lo, hi = ic_bootstrap(d)
            synthese.append({'scenario': sc, 'fraction': frac, 'condition': lib, 'AUC_1re': round(piv[lib].mean(), 4),
                             'gain_vs_aucun': round(d.mean(), 4), 'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4)})
        synthese.append({'scenario': sc, 'fraction': frac, 'condition': 'aucun', 'AUC_1re': round(base.mean(), 4), 'gain_vs_aucun': 0.0})
        for h_, a, b in [('couverture', 'expert_c100', 'expert_c25'), ('justesse', 'expert_c100', 'theme_faux_c100'),
                         ('proximité', 'theme_faux_c100', 'hors_theme_faux_c100'), ('sens des flèches', 'expert_c100', 'inverse_c100'),
                         ('graphe appris vs expert', 'appris_sans_prior', 'expert_c100')]:
            d = piv[a] - piv[b]
            hyp.append({'scenario': sc, 'fraction': frac, 'effet': h_, 'comparaison': f'{a} − {b}', 'ecart_moyen': round(d.mean(), 4),
                        'graines_positives': f'{int((d > 0).sum())}/{len(d)}', 'p': stats.ttest_1samp(d, 0).pvalue if d.std() > 0 else np.nan})
for sc in SCENARIOS:
    for frac in [1.0] + FRACTIONS:
        s = tab[(tab.scenario == sc) & (tab.fraction == frac)]
        for nom, colonne_auc in [('appris (flèches apprises)', 'appris_AUC'), ('ordre de passage', 'ordre_AUC'), ('co-réussite (lift)', 'lift_AUC'),
                                 ('pratique avant (A avant B aide B)', 'pratique_AUC')]:
            v = s[colonne_auc].dropna()
            lo, hi = ic_bootstrap(v) if v.std() > 0 else (v.mean(), v.mean())
            recup.append({'scenario': sc, 'fraction': frac, 'methode': nom, 'AUC_vs_vrais_prerequis': round(v.mean(), 4),
                          'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4),
                          'precisionK': round(s[colonne_auc.replace('AUC', 'precisionK')].dropna().mean(), 4)})
synthese, hyp, recup = pd.DataFrame(synthese), pd.DataFrame(hyp), pd.DataFrame(recup)
o = hyp['p'].dropna().sort_values(); m = len(o)
hyp['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(hyp.index)
hyp['significatif_holm_5%'] = hyp['p_holm'] < 0.05
hyp[['p', 'p_holm']] = hyp[['p', 'p_holm']].astype(float).round(4)
for nom_, t_ in [('sim_synthese', synthese), ('sim_hypotheses', hyp), ('sim_recuperation', recup)]:
    t_.to_csv(os.path.join(RESULTATS, f'{nom_}.csv'), index=False)
print('===== KT : gain d\'AUC (1res tentatives) par rapport à « aucun » =====\n', synthese.to_string(index=False))
print('\n===== Effets (bilatéraux, Holm sur', m, 'tests) =====\n', hyp.to_string(index=False))
print('\n===== Récupération des vrais prérequis (AUC, 0,5 = hasard) =====\n',
      recup.pivot_table(index=['scenario', 'fraction'], columns='methode', values='AUC_vs_vrais_prerequis').to_string())

NOMS = {'expert_c25': 'expert\n25 %', 'expert_c100': 'expert\n100 %', 'theme_faux_c100': 'faux même\nthème', 'hors_theme_faux_c100': 'faux autre\nthème',
        'inverse_c100': 'sens\ninversé', 'appris_sans_prior': 'appris'}
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, sc in enumerate(SCENARIOS):
    ax = axes[0, j]; x = np.arange(len(NOMS))
    for frac, dec in [(0.2, -0.2), (0.05, 0.2)]:
        s = synthese[(synthese.scenario == sc) & (synthese.fraction == frac)].set_index('condition').loc[list(NOMS)]
        err = np.vstack([s.gain_vs_aucun - s.IC95_bas, s.IC95_haut - s.gain_vs_aucun]).astype(float).clip(min=0)
        ax.bar(x + dec, s.gain_vs_aucun.astype(float), width=0.4, yerr=err, capsize=2, label=f'{frac:.0%} des élèves')
    ax.set_xticks(x); ax.set_xticklabels(NOMS.values(), fontsize=7); ax.axhline(0, c='black', lw=0.8)
    ax.set_title(f'{sc} ({"prérequis forts" if sc == "S1" else "prérequis faibles" if sc == "S2" else "ordre imposé, sans prérequis"}) — KT')
    ax2 = axes[1, j]; r_ = recup[recup.scenario == sc]
    ordre_m = ['appris (flèches apprises)', 'ordre de passage', 'co-réussite (lift)', 'pratique avant (A avant B aide B)']
    for k, (frac, dec) in enumerate([(1.0, -0.27), (0.2, 0.0), (0.05, 0.27)]):
        t = r_[r_.fraction == frac].set_index('methode').loc[ordre_m]
        ax2.bar(np.arange(len(t)) + dec, t.AUC_vs_vrais_prerequis, width=0.27, label=f'{frac:.0%} des élèves',
                yerr=np.vstack([t.AUC_vs_vrais_prerequis - t.IC95_bas, t.IC95_haut - t.AUC_vs_vrais_prerequis]).clip(min=0), capsize=2)
    ax2.set_xticks(np.arange(4)); ax2.set_xticklabels(['appris', 'ordre de\npassage', 'co-réussite', 'pratique\navant'], fontsize=8)
    ax2.axhline(0.5, ls=':', c='black'); ax2.set_ylim(0.3, 1.0); ax2.set_title(f'{sc} — récupération des prérequis')
axes[0, 0].set_ylabel("Gain d'AUC (1res tentatives)\npar rapport à « aucun »"); axes[0, 0].legend(fontsize=7)
axes[1, 0].set_ylabel('AUC face aux vrais prérequis'); axes[1, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(RESULTATS, 'sim_resultats.png'), dpi=200); plt.show()

In [ ]:
# Cellule S6 — Rassembler les résultats de l'étape 17c dans un zip et le télécharger
import zipfile
FICHIERS = ['sim_resultats_A.csv', 'sim_resultats_B.csv', 'sim_synthese.csv', 'sim_hypotheses.csv', 'sim_recuperation.csv',
            'sim_factoriel_graphes.csv', 'sim_resultats.png']
chemin_zip = '/content/resultats_etape17c.zip'
with zipfile.ZipFile(chemin_zip, 'w') as z:
    for f in FICHIERS:
        c = os.path.join(RESULTATS, f)
        if os.path.exists(c):
            z.write(c, arcname=f); print('ajouté   :', f)
        else:
            print('MANQUANT :', f)
try:
    from google.colab import files
    files.download(chemin_zip)
except ImportError:
    print('Hors de Colab : le fichier est ici ->', chemin_zip)